# Studying the $\Phi/\Psi$ angle pair of "alanine dipeptide"

In this unit, we are working with "alanine dipeptide", an alanine in which both the N- and the C-terminus are modified with neutral blocking groups (both ends are effectively amides). This is the smallest system which contains the equivalent of the $\Phi/\Psi$ angle pair of peptide and protein backbones, and it is an often used model system in our field. In the simulation community, it is referred to as alanine dipeptide, though technically it is just a single amino acid.

The scripts provided are flexible, so you can work with other amino acids. I strongly recommend, however, to start with alanine first. We are going to cover the following topics

- Setup system
- Make a systematic scan of $\Phi/\Psi$ using energy minimization
- Run MD and
- MC and see whether all relevant regions (according to the systematic energy map are visited)

The goal of this unit is to give an example where one can compare minimization, MC and MD in a somewhat meaningful way.

### 1. Setting up the system

Start a terminal window and make sure that you are in *this* directory.

For all of the following steps, we need to prepare alanine dipeptide for computation. We are (at least in the
first pass) going to work with the CHARMM 36 force field. Carry out the
following commands in the terminal window:

    charmm aa:ala ff:36 -i charmm/setup_aa.inp > charmm/setup_aa.ala.36.out

This generates starting coordinates for alanine dipeptide and writes a PSF, which will 
be reused in all of the following steps. (Note: how you name output file names is up to you, the above is a suggestion!)

*Once things work  for alanine, you can try other amino acids and force fields.
Suppose you want to generate a map for Trp and charmm 27. Carry out the
following commands in the terminal window:*

    charmm aa:trp ff:36 -i charmm/setup_aa.inp > charmm/setup_aa.trp.36.out
    charmm aa:trp ff:27 -i charmm/setup_aa.inp > charmm/setup_aa.trp.27.out

*(Yes, for stupid technical reasons you need both commands if you work with one of the older force fields!)*
*Again, <font color="red">for the time being, stick with Alanine (aa:ala)</font>!!!*


### 2. Creating the $\Phi/\Psi$ energy map

To calculate the $\Phi/\Psi$ energy map, execute

    charmm aa:ala ff:36 -i charmm/phi-psi-1.inp > charmm/phi-psi-1.ala.36.out

This command will run for several minutes!! (We are carrying out
$24\times24$ minimizations, each running for 10,000 steps) Once the job
is finished (once you have again a console prompt), there should be a
file `data/ala.charmm36.pot`, which you can inspect with an editor if
you want. 

You can either download this file an work with it using your preferred software, or
use R available in the notebook. In the latter case, read on:

We need to reformat this file before it can be used by R. One way to do this is 
using the awk utility available on essentially any Unix/Linux system. The
command

    awk -f conv_to_r.awk data/ala.charmm36.pot > data/ala.charmm36.rdat

accomplishes this. The program `awk` is scripting/programming language
specially suited to parsing and (re)formatting text. For this particular
case I find it easier to bring the data into the needed form rather than
trying to do everything with R.

Now you can use the above R commands to read in `data/ala.charmm36.rdat`
and plot a $\Phi$/$\Psi$ map for Ala and CHARMM 36.

In [ ]:
data<-read.table("data/ala.charmm36.rdat") # (A)
x=seq(-180,180,length.out=nrow(data)) # (B)
y=x
z=as.matrix(data) # (C)
levs=c(0.2,0.5,1,2,4,8,16) # (D)
opar=par(no.readonly=T) # (E.1)
par(pin=c(4,4)) # (E.2)
image(x,y,z,col=topo.colors(24),
      xlab=expression(phi),
      ylab=expression(psi))
contour(x,y,z,levels=levs,add=T)
title("Ala, CHARMM 36")
par=opar # (E.3)

Some explanations:

-   1.  The raw data (energy as a function of $\Phi$/$\Psi$) is read in
        first; it’s a quadratic matrix of energy values.

-   1.  We next have to generate vectors containing the $\Phi$, $\Psi$
        values at which these data were generated; I leave it to you to
        figure out the `seq()` command.

-   1.  `read.table` has generated a data frame, but we need to tell R
        that the data contained in `data` really are a *matrix* – needed
        for several types of numerical computations.

-   1.  We specify a vector at which contour lines will be generated.

-   1.  We save the current properties of the plot “device” (E.1) and
        specify a quadratic size (E.2); when we are done we restore the
        default props (E.3).

`image()` generates the colored density map; `contour(..,add=T)`
overlays it with contour lines at the desired levels.

The somewhat convoluted number of steps needed to get to a plot
illustrated a “fact of life” of using computers in computational
chemistry and biology: There, usually, is no single tool/program that
does everything perfectly and some “ducting” (such as the `awk` command)
are required to combine the outputs of several programs in a meaningful
way.

**Before you continue:** Take a look at the plot and identify the regions of local minima. This energy scan is our reference for the next steps.

### 3. Running MD

In this and the next section, we'll test whether a MD or MC simulation samples all minima identified by the systematic scan. As a first step, run

    charmm aa:ala ff:36 -i charmm/pept_md.inp > charmm/pept_md.ala.36.out
    
Again, this will take a few minutes (we run 5M MD steps.)

The MD job saves a trajectory, from which we extract timeseries of $\Phi$ and $\Psi$ values by
running.

    charmm aa:ala ff:36 -i charmm/pept_ana.inp
    
Take a look at the raw output in <tt>./data/phi_psi_md.ala_36.dat</tt>. Either download the file and plot/visualize it locally, *or* process with R

In [ ]:
mddat<-read.table("data/phi_psi_md.ala_36.dat") # read data

In [ ]:
plot(mddat$V1,mddat$V2,ylim=c(-180,180)) #V1 is time, $V2 is phi -- timeseries of phi vs t

In [ ]:
plot(mddat$V2,mddat$V3,xlim=c(-180,180),ylim=c(-180,180)) #V3 is psi, $V2 is phi -- phi/psi scatter plot

As you can see above, a 'short' MD simulation does not visit the second local minimum. If you are willing to wait, you can extend the simulation. You can either edit the input files (change NSTEP) or use the following mechanism:


     charmm aa:ala ff:36 extend:10 -i charmm/pept_md.inp > charmm/pept_md.ala.36.out
     charmm aa:ala ff:36 skip:10 -i charmm/pept_ana.inp
     
The scripts "interpret" the additional parameters as extending the simulation length by 10, but skipping every tenth value during post-processing. (This is not any magic provided by CHARMM; rather CHARMM inputs are simple "programs", and based on these variables based from the outside the behavior of the "program" (CHARMM input) is altered. WARNING: With "extend:10" the first script will run for at least 30 minutes. If a job of this length does not finish and is 'killed', let me know, then I have to contact ZID (the university's central IT department) to fiddle with some settings.

You may want to put the job in the background, ie.

     charmm ... > output & # note the ampersand &
     
You get the command prompt back immediately, with <tt>ps</tt> you can see that a <tt>charmm</tt> job is running. *Obviously, you can only start the post-processing job after the job has finished.*

### 4. Running MC

Analogous scripts are provided for MC. To generate a trajectory and post-process it, run

    charmm aa:ala ff:36 -i charmm/pept_mc.inp > charmm/pept_mc.ala.36.out
    charmm aa:ala ff:36 mc:1 -i charmm/pept_ana.inp

Note: MD and MC are not directly comparable; in the default configuration (without passing <tt>extend</tt> the script carries out 5M MC moves; this is less than 5M MD steps, in each of which all atoms of the system move!)

In [ ]:
mcdat<-read.table("data/phi_psi_mc.ala_36.dat") # read data

In [ ]:
plot(mcdat$V1,mcdat$V2,ylim=c(-180,180)) #V1 is time, $V2 is phi -- timeseries of phi vs t

NOTE: Time has no meaning in MC; you really get the number of moves

In [ ]:
plot(mcdat$V2,mcdat$V3,xlim=c(-180,180),ylim=c(-180,180)) #V3 is psi, $V2 is phi -- phi/psi scatter plot

As in MD, the default length of the MC does not sample the secondary minimum (at least when I run this). If you have time, you can again try adding the <tt>extend:10</tt> parameter (which will increase the run length by a factor of 10! (For me, the secondary minimum does get sampled in the longer run!)

<font color="red">**TODO/EXERCISE:**</font> Please upload a short description of what you did / tried (which calculations did you run), with illustrative plots (either in this notebook, if you use R, or upload some plots, if, e.g., you worked with Excel). <font color="red">Please "assemble" your "protocol" into **a single PDF** file</font> -- if you use the Jupyter Notebook, use the 'Export Notebook as' functionality!

The main takehome message should be that both MD and MC need significant time to find the second, local minimum; this is why often various techniques to enhance sampling are used. Alanine "dipeptide" is the zeroth order test system for such methods!